# 05 · Fase 4: iniciar detector de lesiones TB

YOLO11n pretrained COCO, con una clase tb_lesion. Convierte las cajas originales
de train/validation a YOLO manteniendo sus particiones. Health/sick son negativos
sin cajas. TB sin cajas se excluye: no es un negativo. Los subtipos se conservan
en los metadatos originales, pero no se exigen como clases de detección.

**Fase 4 preparada para iniciar, no ejecutada en esta fuente.** Este notebook
no contiene la evaluación final de Fase 5. El dataset.yaml no define test.
Ultralytics 8.3.228 se fija para reproducibilidad; su código se distribuye bajo
AGPL-3.0. Revise esa licencia al distribuir el sistema posterior.

## Montar Drive y recuperar el paquete nuevo

Abra este notebook en Colab con GPU L4/A100/T4 disponible. El código se extrae
en otra carpeta temporal; los notebooks ejecutados 02/03 y sus resultados se
conservan. El nuevo ZIP debe estar en MyDrive/Proyecto_Tuberculosis/.
Guarde las salidas de cada notebook con Ctrl+S.

In [ ]:
import json, sys, hashlib, stat, zipfile, subprocess
from pathlib import Path, PurePosixPath
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BASE = Path('/content/drive/MyDrive/Proyecto_Tuberculosis')
ARCHIVE = DRIVE_BASE / 'tbx11k_fases_3_y_4_colab.zip'
PROJECT_ROOT = Path('/content/tbx11k_complete/project')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as z:
    for item in z.infolist():
        rel = PurePosixPath(item.filename)
        target = (PROJECT_ROOT / item.filename).resolve()
        if rel.is_absolute() or '..' in rel.parts or ':' in item.filename or '\\' in item.filename or stat.S_ISLNK(item.external_attr >> 16) or PROJECT_ROOT.resolve() not in target.parents:
            raise ValueError('Ruta no segura: '+item.filename)
    checksums = json.loads(z.read('phase3_package_checksums.json'))
    for name, expected in checksums.items():
        if hashlib.sha256(z.read(name)).hexdigest() != expected:
            raise ValueError('Archivo alterado en el paquete: '+name)
    z.extractall(PROJECT_ROOT)
# Importa el código nuevo incluso si el runtime había cargado el paquete anterior.
for name in list(sys.modules):
    if name in {'tbxtrain','tbxdata'} or name.startswith(('tbxtrain.','tbxdata.')):
        del sys.modules[name]
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                       '--disable-pip-version-check', '-r', str(PROJECT_ROOT / 'requirements-detection.txt')])
import torch
if not torch.cuda.is_available(): raise RuntimeError('Seleccione un runtime GPU de Colab.')
from tbxdata.colab import stage_dataset
from tbxdata.validate import validate_manifests
source = DRIVE_BASE / 'TBX11K.zip'
if not source.exists(): source = DRIVE_BASE / 'TBX11K'
DATASET_ROOT = stage_dataset(source, Path('/content/tbx11k_complete/data_cache'), True)
integrity = validate_manifests(PROJECT_ROOT, DATASET_ROOT, all_image_hashes=True)
print('GPU:',torch.cuda.get_device_name(0),'| Integridad:',integrity['status'])
print('Hashes de imágenes verificados:',integrity['all_image_hashes_verified'])

In [ ]:
from datetime import datetime, timezone
from tbxtrain.detection import prepare_detection
from tbxtrain.protocol import atomic_json
# Para reanudar, escriba el nombre de la carpeta del detector ya creada.
DETECTOR_RESUME_RUN = None
DET_RUN = DETECTOR_RESUME_RUN or ('detector_yolo11n_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
if Path(DET_RUN).name != DET_RUN or any(c in DET_RUN for c in '/\\:') or DET_RUN in {'.','..'}:
    raise ValueError('Nombre de corrida detector inválido.')
DET_DATA = Path('/content/tbx11k_complete') / DET_RUN
if not DET_DATA.exists():
    conversion = prepare_detection(PROJECT_ROOT, DATASET_ROOT, DET_DATA)
else:
    conversion = json.loads((DET_DATA/'conversion.json').read_text())
print(json.dumps(conversion,indent=2,ensure_ascii=False))
DET_OUTPUT = DRIVE_BASE / 'experiments' / DET_RUN
DET_OUTPUT.mkdir(parents=True,exist_ok=bool(DETECTOR_RESUME_RUN))
atomic_json(DET_OUTPUT / 'conversion.json',conversion)

## F4-E1 · Revisar cajas convertidas antes de entrenar

Los rectángulos se dibujan desde las etiquetas YOLO exportadas sobre la imagen
copiada, para comprobar la conversión y las coordenadas. Una revisión técnica
no cambia las anotaciones ni sustituye su revisión por el equipo.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
positive = [p for p in (DET_DATA / 'labels/train').glob('*.txt') if p.stat().st_size][:6]
fig, axes = plt.subplots(2,3,figsize=(12,8))
for ax in axes.flat: ax.axis('off')
for ax,label_path in zip(axes.flat,positive):
    image_path = DET_DATA / 'images/train' / (label_path.stem+'.png')
    with Image.open(image_path) as im:
        w,h=im.size; ax.imshow(im.convert('RGB'))
    for line in label_path.read_text().splitlines():
        cls,cx,cy,bw,bh=map(float,line.split())
        ax.add_patch(patches.Rectangle(((cx-bw/2)*w,(cy-bh/2)*h),bw*w,bh*h,
                                       fill=False,edgecolor='red',linewidth=1.5))
fig.tight_layout(); fig.savefig(DET_OUTPUT/'converted_boxes.png',dpi=150); plt.show()

## F4-E2 · Entrenamiento opcional del detector

Se deja desactivado para que pueda revisar primero la conversión. Al activarlo
se entrena con todo train y se valida con validation. Se desactivan espejado,
mosaico y geometrías agresivas. El detector maneja sus transformaciones de
imagen/cajas conjuntamente. No ajuste estos parámetros con test.

El smoke opcional usa hasta tres positivos y tres negativos por split,
solo como verificación técnica; sus métricas
no sirven como resultado del detector. El entrenamiento real está separado.

In [ ]:
from ultralytics import YOLO
RUN_DETECTOR_SMOKE = False
RUN_DETECTOR_TRAINING = False
common = dict(data=str(DET_DATA/'dataset.yaml'),imgsz=512,batch=8,device=0,workers=2,
              seed=20261019,amp=False,fliplr=0.,flipud=0.,mosaic=0.,mixup=0.,
              degrees=0.,translate=0.,scale=0.,shear=0.,perspective=0.,
              hsv_h=0.,hsv_s=0.,hsv_v=.08,save=True,save_period=1)
if RUN_DETECTOR_SMOKE:
    smoke_common = dict(common,data=str(DET_DATA/'smoke.yaml'),batch=2)
    YOLO('yolo11n.pt').train(**smoke_common,epochs=1,
        project=str(DET_OUTPUT),name='smoke',exist_ok=False)
if RUN_DETECTOR_TRAINING:
    if DETECTOR_RESUME_RUN:
        last = DET_OUTPUT/'training/weights/last.pt'
        if not last.is_file(): raise FileNotFoundError(last)
        YOLO(str(last)).train(resume=True,device=0)
    else:
        if (DET_OUTPUT/'training').exists(): raise RuntimeError('Ya existe training/. Reanude esa corrida o prepare un DET_RUN nuevo.')
        YOLO('yolo11n.pt').train(**common,epochs=35,patience=7,
            project=str(DET_OUTPUT),name='training',exist_ok=False)
else:
    print('Entrenamiento detector pendiente. Conversión y revisión ya pueden documentarse.')

## F4-E3 · Validación y salida del detector

Solo habilitada si existe un entrenamiento real. Conserva best/last.pt,
results.csv, curvas, configuración y mAP de validation. La evaluación final,
Grad-CAM y regla clasificación–detección se preparan después en Fase 5.
Para reanudar una desconexión del detector, escriba su nombre de carpeta en
DETECTOR_RESUME_RUN, repita preparación y active RUN_DETECTOR_TRAINING. Se
reconstruye la misma ruta local y YOLO recupera training/weights/last.pt.

In [ ]:
if RUN_DETECTOR_TRAINING:
    best = DET_OUTPUT/'training/weights/best.pt'
    if not best.is_file(): raise FileNotFoundError(best)
    trained = YOLO(str(best))
    result = trained.val(data=str(DET_DATA/'dataset.yaml'),split='val',imgsz=512,
                          device=0,project=str(DET_OUTPUT),name='validation',exist_ok=False)
    summary = dict(status='trained_validation_available',
                   map50_95=float(result.box.map),map50=float(result.box.map50),
                   validation_only=True,test_evaluated=False,phase5_complete=False)
    atomic_json(DET_OUTPUT/'detector_summary.json',summary)
    print(summary)